# Automated Candidate Interview & Evaluation System

This notebook demonstrates an AI-powered interview system using AutoGen. It features three agents:

- **Interviewer** — Asks structured interview questions
- **Candidate** — A human-in-the-loop proxy that takes real user input
- **Evaluator** — Provides feedback and coaching on the candidate's responses

## Setup & Imports

In [3]:
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.base import TaskResult
from dotenv import load_dotenv

import os
load_dotenv()


GROQ_API_KEY = os.getenv("GROQ_API_KEY")

config_list = {
    "config_list": [{"model": "llama-3.3-70b-versatile", "temperature": 0.1}]
}

model_client = OpenAIChatCompletionClient(
    model=config_list["config_list"][0]["model"],
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "family": "unknown",
    },
)

## Define the Interview Agents & Team

In [4]:
async def my_agents(job_position = "AI Engineer"):
    interviewer = AssistantAgent(
        name = "Interviewer",
        model_client= model_client,
        description=f"An AI agent that conducts interviews for a {job_position} position.",
        system_message=f'''
        You are a professional interviewer for a {job_position} position.
        Ask one clear question at a time and Wait for user to respond. 
        Your job. is to continue and ask questions, don't pay any attention to career coach response. 
        Make sure to ask question based on Candidate's answer and your expertise in the field.
        Ask 3 questions in total covering technical skills and experience, problem-solving abilities, and cultural fit.
        After asking 3 questions, say 'TERMINATE' at the end of the interview.
        Make question under 50 words.
    '''
    )

    candidate = UserProxyAgent(
        name= "Candidate",
        description=f"An agent that simulates a candidate for a {job_position} position.",
        input_func=input
    )

    evaluation = AssistantAgent(
        name = "Evaluator",
        model_client= model_client,
        description=f"An AI agent that provides feedback and advice to candidates for a {job_position} position.",
        system_message=f'''
        You are a career coach specializing in preparing candidates for {job_position} interviews.
        Provide constructive feedback on the candidate's responses and suggest improvements.
        After the interview, summarize the candidate's performance and provide actionable advice.
        Make it under 100 words.
    '''
    )

    terninate_condition = TextMentionTermination(text="TERMINATE")

    team = RoundRobinGroupChat(
        participants=[interviewer, candidate, evaluation],
        termination_condition= terninate_condition,
        max_turns=20
    )

    return team

## Stream the Interview

In [5]:
async def run_interview(team):
    async for message in team.run_stream(task='Start the interview with the first question ?'):
        if isinstance(message, TaskResult):
            message = f'Interview completed with result: {message.stop_reason}'
            yield message
        else:
            message = f'{message.source}: {message.content}'
            yield message

## Run the Interview

In [6]:
job_position = "AI Engineer"
team = await my_agents(job_position)

async for message in run_interview(team):
    print('-' * 70)
    print(message)

----------------------------------------------------------------------
user: Start the interview with the first question ?
----------------------------------------------------------------------
Interviewer: What programming languages do you use for AI development?
----------------------------------------------------------------------
Candidate: 
----------------------------------------------------------------------
Candidate: Python
----------------------------------------------------------------------
Evaluator: That's a great start. Python is a popular and versatile language for AI development. Can you elaborate on why you prefer Python and how you've used it in AI-related projects? Additionally, are you familiar with other languages used in AI, such as R or Julia?
----------------------------------------------------------------------
Interviewer: How do you handle overfitting in machine learning models?
----------------------------------------------------------------------
Candidate